In [ ]:
# 07_explain_breakdown.ipynb
# Post-hoc Explainability using BreakDown

import sys
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_INSTANCES


import pandas as pd
import numpy as np
import joblib
import dalex as dx
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

processed_path = Path("../datasets/processed")
models_path = Path("../results/black_box_models")
results_path = Path("../results/Breakdown")
results_path.mkdir(parents=True, exist_ok=True)

for ds in DATASETS:
    print(f"\n=== Dataset: {ds} ===")

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    s = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    train_idx, test_idx = next(s.split(X, y))
    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    X_test  = X.iloc[test_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)

    defect_indices = np.where(y_test.values == 1)[0]
    if len(defect_indices) == 0:
        print(f"  No defective instances in {ds}")
        continue

    ds_model_path = models_path / ds
    ds_result_path = results_path / ds
    ds_result_path.mkdir(parents=True, exist_ok=True)

    for model_dir in sorted(ds_model_path.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        print(f"Model: {model_name}")

        model = joblib.load(model_dir / "model.joblib")
        model_result_path = ds_result_path / model_name
        model_result_path.mkdir(parents=True, exist_ok=True)

        explainer = dx.Explainer(
            model=model,
            data=X_train,
            y=y_train,
            label=model_name,
            verbose=False
        )

        records = []

        for i, idx in enumerate(defect_indices[:N_INSTANCES]):
            instance = X_test.iloc[idx:idx + 1]

            breakdown = explainer.predict_parts(instance, type="break_down")

            if breakdown is None or breakdown.result is None:
                continue

            df_bd = pd.DataFrame(breakdown.result)

            if "contribution" not in df_bd.columns:
                continue

            df_bd = df_bd[~df_bd["variable"].isin(["_baseline_", "_full_model_"])]
            contribs = df_bd["contribution"].astype(float).values

            model_pred = int(model.predict(instance)[0])
            prob_defect = float(model.predict_proba(instance)[0, 1])

            plot_obj = breakdown.plot(show=False)
            if plot_obj is not None:
                plot_obj.write_html(
                    str(model_result_path / f"instance_{i}_breakdown.html")
                )